In [9]:
from pathlib import Path
import sqlite3

import pandas as pd
import pendulum

from kibad_crawler.config import RAW_DATA_DIR

# pyparsing exploration

In [3]:
from pyparsing import (
    CaselessLiteral,
    alphanums,
    infix_notation,
    opAssoc,
    quotedString,
    Word,
    removeQuotes,
)

and_ = CaselessLiteral("and")
or_ = CaselessLiteral("or")
not_ = CaselessLiteral("not")
search_term = Word(alphanums) | quotedString.set_parse_action(removeQuotes)
search_expression = infix_notation(
    search_term,
    [
        (not_, 1, opAssoc.RIGHT),
        (and_, 2, opAssoc.LEFT),
        (or_, 2, opAssoc.LEFT),
    ],
)

tests = """\
    wood and blue or red
    wood and (blue or red)
    (steel or iron) and "lime green"
    not steel or iron and "lime green"
    not(steel or iron) and "lime green" 
    ("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*") """.splitlines()

for t in tests:
    print(t.strip())
    print(search_expression.parse_string(t)[0])
    print()

wood and blue or red
[['wood', 'and', 'blue'], 'or', 'red']

wood and (blue or red)
['wood', 'and', ['blue', 'or', 'red']]

(steel or iron) and "lime green"
[['steel', 'or', 'iron'], 'and', 'lime green']

not steel or iron and "lime green"
[['not', 'steel'], 'or', ['iron', 'and', 'lime green']]

not(steel or iron) and "lime green"
[['not', ['steel', 'or', 'iron']], 'and', 'lime green']

("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*")
['species richness', 'or', 'species number*', 'or', 'number of species', 'or', 'species identity', 'or', 'species composition', 'or', 'species abundance', 'or', 'species divers*']



In [5]:
def check_string(text: str):
    print(text.strip())
    print(search_expression.parse_string(text)[0])


def parse_query_string(query_plain_text: str):
    query_parsed = search_expression.parse_string(query_plain_text)[0]
    # for group in query_parsed:
    #     if not isinstance(group, str):
    #         print([x for x in group if x != "or"])
    #     else:
    #         print(group)
    return query_parsed


BIODIVERSITY = """("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*" OR "species evenness" OR diversity OR "biodivers*" OR "biodiversity hotspot*" OR "genetic* divers*" OR "genetic* richness" OR "landscape divers*" OR "landscape structure" OR "landscape composition" OR "habitat divers*" OR "habitat structure" OR "habitat composition" OR "seascape structure" OR "structur* divers*" OR "community composition" OR "ecosystem* divers*" OR "ecosystem* structure" OR "ecosystem composition" OR "phylogenetic* divers*" OR "phylogenetic* richness" OR "number of taxa" OR "taxonomic* divers*" OR "taxonomic* richness" OR "functional* divers*" OR "functional* richness" OR "functional* composition" OR "functional* evenness" OR "functional* dissimilarity" OR "functional* similarity" OR shannon OR simpson OR evenness OR richness OR biomass OR abundance OR density OR "communit*")"""
DEUTSCHLAND = """("German*" OR "central europ*" OR "Lower Saxony" OR Thuringia OR Saarland OR "Berlin*" OR "Brandenburg*" OR "Saxon*" OR "Saxony-Anhalt" OR "Bavar*" OR "Baden-Wuerttemberg" OR Bremen OR Hamburg OR "Hess*" OR "Mecklenburg-Western Pomerania" OR "North RhineWestphalia" OR "Rhineland-Palatinate" OR "Schleswig-Holstein")"""
LEBENSRAUM = """("Forest*" OR "wood*" OR "stock of trees" OR stand OR stands OR "plantation*" OR "maple stand*" OR "birch stand*" OR "beech stand*" OR "rowan stand*" OR "yew stand*" OR "oak stand*" OR "alder stand*" OR "ash stand*" OR "spruce stand*" OR "hornbeam stand*" OR "hazel stand*" OR "chestnut stand*" OR "pine stand*" OR "larch stand*" OR "linden stand*" OR "poplar stand*" OR "sloe stand*" OR "fir stand*" OR "willow stand*" OR "hawthorne stand*" OR "elm stand*" OR deciduous OR coniferous OR "mixed forest*" OR "mixed woodland*" OR coppice OR copse)"""
BASE_KEYWORDS = BIODIVERSITY + " AND " + DEUTSCHLAND + " AND " + LEBENSRAUM
TRENDS = (
    BASE_KEYWORDS
    + " AND "
    + """("trend*" OR "chang*" OR "decreas*" OR "gain*" OR "increas*" OR "los*" OR "loss*" OR "reduc*" OR "declin*" OR "extinct*" OR "develop*" OR "species loss" OR "biodiversity loss" OR "grow*" OR "endangered species" OR "extinct species")"""
)

query_parsed = parse_query_string(TRENDS)
for group in query_parsed:
    if not isinstance(group, str):
        print([x for x in group if x != "or"])
    else:
        print(group)

['species richness', 'species number*', 'number of species', 'species identity', 'species composition', 'species abundance', 'species divers*', 'species evenness', 'diversity', 'biodivers*', 'biodiversity hotspot*', 'genetic* divers*', 'genetic* richness', 'landscape divers*', 'landscape structure', 'landscape composition', 'habitat divers*', 'habitat structure', 'habitat composition', 'seascape structure', 'structur* divers*', 'community composition', 'ecosystem* divers*', 'ecosystem* structure', 'ecosystem composition', 'phylogenetic* divers*', 'phylogenetic* richness', 'number of taxa', 'taxonomic* divers*', 'taxonomic* richness', 'functional* divers*', 'functional* richness', 'functional* composition', 'functional* evenness', 'functional* dissimilarity', 'functional* similarity', 'shannon', 'simpson', 'evenness', 'richness', 'biomass', 'abundance', 'density', 'communit*']
and
['German*', 'central europ*', 'Lower Saxony', 'Thuringia', 'Saarland', 'Berlin*', 'Brandenburg*', 'Saxon*',

In [15]:
files_path = Path(RAW_DATA_DIR / "semantic_scholar" / "release" / "2026-08-25" / "papers")
json_files = files_path.glob("**/*.json")

In [7]:
# query_parsed[0]
processing_steps = []
for group in query_parsed:
    if not isinstance(group, str):
        # print([x for x in group if x != "or"])
        processing_steps.append([x for x in group if x != "or"])
    else:
        # print(group)
        processing_steps.append(group)

processing_or_steps = [x for x in processing_steps if isinstance(x, list)]

In [16]:
total_papers = 0
start = pendulum.now()
for json_file in json_files:
    df = pd.read_json(json_file)
    df = df[["paperId", "title", "abstract"]]
    total_papers += df.shape[0]

    for i, r in df.iterrows():
        text = f"{r['title'].strip()} {r['abstract'].strip() if r['abstract'] else ""}"

        check_result = []
        for step in processing_or_steps:
            check_result.append(
                any(word.replace("*", "").lower() in text.lower() for word in step)
            )

        result = all(check_result)
        if result == True:
            print(r["paperId"])
            # print(f"{i}, {text}")
end = pendulum.now()
print(f"{total_papers:,} papers in {(end - start).in_words()}")

8cfe395192d4485f68fbfbb3d5511b1beadbea41
811056fa7df79f4d869f5dc4c0904e38531b73e8
42f621a6a6ed330fd36738ebd0b9456954d2543a
ce5df17ea5b7a0a0c1bec186405a5527abb815a4
1c699909c19f60b14458cdc73ab12f8cb49a031a
588cfc9345a3e8f2ae03dae165b0442952c63d18
3a09eaeb4c9e25190842a603691edb5ef576d601
93666f5f4ed52e732127b053b9795cefb2158efc
aa760ac9761e69ea405c98da5883d387d71fc993
cb6d92894a008d43a08f8267d3ba80de7ac4a054
c7e1d539bdfd64d5cad24b694db3f62d13b2b24c
7ddff258b09833610d007445551ee34332824284
623b3191720f01d32a48311239e9eaa24fbbb9e2
f8d906673c3fb0e2c62ad823411febc0218001ac
02e3f72d8df8e57e70cd9a541b74563dd8a55bf3
20e4d303a74f71d4c2c34b3a543e8dd8566f9736
1297b2edc5c51bb27044f2b26dc79c66a61fed9d
c4466c14058a9dfd6ee5c8941128aa21058015b8
1a5210e7a34b5473ae3ed2997270678fcd5bac44
52074242fbf2cb0d821de01c4f4a3c8164195233
d36fdf2ab0f4284eae0280aa426ddb8cb3a829b5
357bf17dfd3a2fe2d452b9555bcf90038dde069a
e4d176bd3fc633bdb545dc432ebf16f2b796d7ac
1dad947f206b37d96a9616db689e6aacc7347338
22a90f47292b7878

In [ ]:
"""As https://pypi.org/project/sifts/ is leveraging SQLite3 FTS5 capabilities
the crawler can be based on them.

- How to represent queries
- How to load queries
- How to represent '*' and '?' from the words
"""

# https://pypi.org/project/sifts/
# https://www.sqlite.org/fts5.html
# https://medium.com/@johnidouglasmarangon/full-text-search-in-sqlite-a-practical-guide-80a69c3f42a4

# SQLite3 exploration

In [8]:
def get_db():
    return sqlite3.connect("papers.db")

In [3]:
conn = get_db()
cursor = conn.cursor()
cursor.execute("CREATE VIRTUAL TABLE papers USING fts5(title, abstract);")
conn.commit()

In [4]:
cursor.execute("SELECT * FROM papers")
cursor.fetchall()

[]

In [ ]:
query_create = """CREATE VIRTUAL TABLE papers USING fts5(title, abstract);"""

In [13]:
query = """("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*" OR "species evenness" OR diversity OR biodivers* OR "biodiversity hotspot*" OR "genetic* divers*" OR "genetic* richness" OR "landscape divers*" OR "landscape structure" OR "landscape composition" OR "habitat divers*" OR "habitat structure" OR "habitat composition" OR "seascape structure" OR "structur* divers*" OR "community composition" OR "ecosystem* divers*" OR "ecosystem* structure" OR "ecosystem composition" OR "phylogenetic* divers*" OR "phylogenetic* richness" OR "number of taxa" OR "taxonomic* divers*" OR "taxonomic* richness" OR "functional* divers*" OR "functional* richness" OR "functional* composition" OR "functional* evenness" OR "functional* dissimilarity" OR "functional* similarity" OR shannon OR simpson OR evenness OR richness OR biomass OR abundance OR density OR communit*)"""
print(query)

("species richness" OR "species number*" OR "number of species" OR "species identity" OR "species composition" OR "species abundance" OR "species divers*" OR "species evenness" OR diversity OR biodivers* OR "biodiversity hotspot*" OR "genetic* divers*" OR "genetic* richness" OR "landscape divers*" OR "landscape structure" OR "landscape composition" OR "habitat divers*" OR "habitat structure" OR "habitat composition" OR "seascape structure" OR "structur* divers*" OR "community composition" OR "ecosystem* divers*" OR "ecosystem* structure" OR "ecosystem composition" OR "phylogenetic* divers*" OR "phylogenetic* richness" OR "number of taxa" OR "taxonomic* divers*" OR "taxonomic* richness" OR "functional* divers*" OR "functional* richness" OR "functional* composition" OR "functional* evenness" OR "functional* dissimilarity" OR "functional* similarity" OR shannon OR simpson OR evenness OR richness OR biomass OR abundance OR density OR communit*)


In [25]:
"""
- capture strings
- identify startswith, endswith or normal
- list of words
"""

# print("AND" in query)
# print(query.replace('"', "").replace("(", "").replace(")", "").split(" OR "))
words = query.replace('"', "").replace("(", "").replace(")", "").split(" OR ")
for word in words:
    if "*" in word:
        print(f"{word} | {word.count('*')} * | {word.count(' ')} words")
    else:
        print(word)

species richness
species number* | 1 * | 1 words
number of species
species identity
species composition
species abundance
species divers* | 1 * | 1 words
species evenness
diversity
biodivers* | 1 * | 0 words
biodiversity hotspot* | 1 * | 1 words
genetic* divers* | 2 * | 1 words
genetic* richness | 1 * | 1 words
landscape divers* | 1 * | 1 words
landscape structure
landscape composition
habitat divers* | 1 * | 1 words
habitat structure
habitat composition
seascape structure
structur* divers* | 2 * | 1 words
community composition
ecosystem* divers* | 2 * | 1 words
ecosystem* structure | 1 * | 1 words
ecosystem composition
phylogenetic* divers* | 2 * | 1 words
phylogenetic* richness | 1 * | 1 words
number of taxa
taxonomic* divers* | 2 * | 1 words
taxonomic* richness | 1 * | 1 words
functional* divers* | 2 * | 1 words
functional* richness | 1 * | 1 words
functional* composition | 1 * | 1 words
functional* evenness | 1 * | 1 words
functional* dissimilarity | 1 * | 1 words
functional* simil